# 2. Camada Bronze

A camada Bronze representa a primeira etapa de persistência da Arquitetura Medalhão, mantendo os dados ingeridos o mais próximo possível de sua estrutura de origem.

## 2.1 Objetivo da camada Bronze

O objetivo desta etapa é realizar a ingestão dos arquivos históricos de produção offshore da ANP, armazenados no Volume `anp_raw`, e persistir os dados em **tabelas Delta** na camada Bronze.

Os arquivos abrangem o período de **2005 a 2025** e, conforme identificado durante a exploração realizada no notebook anterior, possuem a mesma estrutura lógica, embora apresentem algumas diferenças técnicas de formatação ao longo do histórico.

Nesta camada não serão realizadas transformações de limpeza, padronização ou consolidação dos dados. Essas operações serão executadas posteriormente na camada Silver.

Ao final desta etapa, os 11 arquivos históricos estarão persistidos individualmente em tabelas Delta, constituindo uma base estruturada e rastreável para as etapas seguintes do pipeline.

## 2.2 Preparação da camada Bronze

Para organizar as tabelas de acordo com a Arquitetura Medalhão, será criado no Unity Catalog um schema específico denominado `bronze`.

O schema funcionará como um agrupamento lógico das tabelas pertencentes a essa camada, separando-as dos arquivos originais armazenados no Volume `anp_raw`.

A organização inicial do Lakehouse será, portanto:

- `workspace.default.anp_raw`: armazenamento dos arquivos CSV originais;
- `workspace.bronze`: armazenamento das tabelas Delta correspondentes à camada Bronze.

As tabelas da camada Bronze serão armazenadas no formato **Delta**, utilizado pelo Databricks para persistência de dados no Lakehouse. Esse formato adiciona recursos de gerenciamento e transações sobre os dados armazenados, além de permitir o controle de versões das tabelas e facilitar seu processamento nas etapas seguintes do pipeline.

In [0]:
# Cria o schema destinado às tabelas da camada Bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

DataFrame[]

## 2.3 Configuração da ingestão

Os dados históricos utilizados no projeto estão distribuídos em 11 arquivos CSV, correspondentes a diferentes períodos entre 2005 e 2025.

Para evitar a repetição de código durante a ingestão, será utilizada uma configuração parametrizada contendo, para cada arquivo:

- o nome do arquivo de origem;
- o nome da tabela correspondente na camada Bronze;
- a codificação de caracteres necessária para sua leitura;
- a necessidade de aplicação de uma regra específica de normalização antes do parsing do CSV.

Durante a exploração dos dados, foi identificado que os arquivos referentes aos períodos de **2016–2018, 2019 e 2021** necessitam da codificação `ISO-8859-1` para que os caracteres acentuados sejam interpretados corretamente.

Posteriormente, durante a avaliação de qualidade realizada no notebook `03_qualidade_dados`, especificamente no **tópico 3.2.4 — Consistência dos atributos categóricos**, foram identificadas cardinalidades anômalas nas tabelas referentes aos anos de **2019, 2020 e 2021**.

A investigação dos arquivos de origem mostrou que esses três arquivos possuem milhares de registros nos quais a linha completa está delimitada externamente por aspas, fazendo com que campos internos já delimitados apresentem aspas duplicadas.

Testes realizados diretamente sobre os arquivos RAW demonstraram que a normalização desse padrão, antes da interpretação do conteúdo como CSV, restabelece uma estrutura categórica coerente e preserva a quantidade original de registros.

Por esse motivo, os arquivos de 2019, 2020 e 2021 receberão uma regra específica de leitura durante a ingestão na camada Bronze. Os arquivos originais armazenados na camada RAW permanecerão inalterados.

A parametrização permite tratar as diferenças técnicas identificadas entre os períodos dentro de uma única lógica de processamento, mantendo o pipeline organizado, rastreável e reutilizável.

In [0]:
# Caminho do Volume onde estão armazenados os arquivos originais
caminho_raw = "/Volumes/workspace/default/anp_raw/"

# Configuração dos arquivos que serão ingeridos na camada Bronze.
# O último parâmetro identifica os arquivos de 2019 a 2021, nos quais
# a análise de qualidade detectou registros delimitados externamente
# por aspas que exigem normalização antes do parsing do CSV.
arquivos_bronze = [
    ("producao-mar-2005-2009.csv", "producao_mar_2005_2009", None, False),
    ("producao-mar-2010-2012.csv", "producao_mar_2010_2012", None, False),
    ("producao-mar-2013-2015.csv", "producao_mar_2013_2015", None, False),
    ("producao-mar-2016-2018.csv", "producao_mar_2016_2018", "ISO-8859-1", False),
    ("producao-mar-2019.csv", "producao_mar_2019", "ISO-8859-1", True),
    ("producao-mar-2020.csv", "producao_mar_2020", None, True),
    ("producao-mar-2021.csv", "producao_mar_2021", "ISO-8859-1", True),
    ("producao-mar-2022.csv", "producao_mar_2022", None, False),
    ("producao-mar-2023.csv", "producao_mar_2023", None, False),
    ("producao_por_poco_2024.csv", "producao_mar_2024", None, False),
    ("producao-mar-2025.csv", "producao_mar_2025", None, False)
]

## 2.4 Ingestão dos arquivos históricos

A partir da configuração definida anteriormente, os arquivos serão percorridos de forma automatizada para realizar sua ingestão na camada Bronze.

Para cada arquivo, o processo executará as seguintes etapas:

1. identificação do arquivo e de suas configurações de leitura;
2. aplicação da codificação de caracteres correspondente, quando necessária;
3. aplicação da regra específica de normalização das aspas nos arquivos de 2019, 2020 e 2021;
4. interpretação dos registros como CSV;
5. persistência dos dados como tabela Delta no schema `workspace.bronze`.

Conforme identificado durante a avaliação de qualidade realizada no notebook `03_qualidade_dados`, tópico **3.2.4 — Consistência dos atributos categóricos**, os arquivos de 2019, 2020 e 2021 possuem registros em que a linha completa está delimitada externamente por aspas. Nesses casos, campos internos já delimitados apresentam aspas duplicadas, ocasionando deslocamento dos valores entre as colunas quando utilizada diretamente a leitura convencional de CSV.

Para esses três arquivos, a ingestão será realizada inicialmente em formato de texto. As linhas afetadas terão as aspas externas removidas e as aspas internas duplicadas restauradas antes da interpretação como CSV. Os demais arquivos continuarão utilizando a leitura convencional.

Essa regra constitui um tratamento técnico de ingestão necessário para que a estrutura dos registros seja corretamente interpretada. Os arquivos originais armazenados na camada RAW permanecem inalterados.

Como a normalização é realizada antes da interpretação do CSV, foi definido explicitamente um schema com os 21 atributos esperados para os arquivos de 2019, 2020 e 2021. Nesses períodos, os valores serão inicialmente persistidos como `string`, evitando que a inferência automática de tipos interfira no processo de reconstrução dos registros. A padronização dos tipos de dados será realizada posteriormente na camada Silver. Nesta etapa, não serão realizadas transformações de negócio, padronizações de nomenclatura ou tratamentos de valores.

A gravação será realizada utilizando o modo **overwrite**. Dessa forma, caso uma tabela Bronze já exista, seu conteúdo será substituído pelos dados da nova execução, evitando a duplicação de registros em eventuais reprocessamentos do pipeline.

Durante a primeira tentativa de persistência, foi identificado que alguns nomes de colunas da fonte contêm caracteres não aceitos por padrão na criação de tabelas Delta. Como o objetivo da camada Bronze é preservar a estrutura dos dados o mais próximo possível da origem, optou-se por não renomear essas colunas nesta etapa. Em vez disso, foi habilitado o recurso **Column Mapping** do Delta Lake, permitindo a persistência dos nomes originais. A padronização dos nomes das colunas será realizada posteriormente na camada Silver.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# Schema utilizado especificamente na leitura dos arquivos de 2019 a 2021.
# A análise de qualidade identificou que esses arquivos precisam ser
# normalizados antes do parsing. Os campos são mantidos como string na
# Bronze e terão seus tipos padronizados posteriormente na camada Silver.
colunas_bronze = [
    "Ano",
    "Mês/Ano",
    "Estado",
    "Bacia",
    "Campo",
    "Poço",
    "Ambiente",
    "Instalação",
    "Produção de Óleo (m³)",
    "Produção de Condensado (m³)",
    "Produção de Gás Associado (Mm³)",
    "Produção de Gás Não Associado (Mm³)",
    "Produção de Água (m³)",
    "Injeção de Gás (Mm³)",
    "Injeção de Água para Recuperação Secundária (m³)",
    "Injeção de Água para Descarte (m³)",
    "Injeção de Gás Carbônico (Mm³)",
    "Injeção de Nitrogênio (Mm³)",
    "Injeção de Vapor de Água (t)",
    "Injeção de Polímeros (m³)",
    "Injeção de Outros Fluidos (m³)"
]

schema_bronze_especial = StructType([
    StructField(coluna, StringType(), True)
    for coluna in colunas_bronze
])

# A função from_csv deste ambiente recebe o schema no formato DDL.
# Essa conversão permite utilizar as 21 colunas definidas acima
# sem depender da existência prévia das tabelas Bronze.
schema_bronze_especial_ddl = schema_bronze_especial.toDDL()


# Percorre a configuração definida para cada arquivo histórico.
for nome_arquivo, nome_tabela, encoding, normalizar_aspas in arquivos_bronze:

    caminho_arquivo = f"{caminho_raw}{nome_arquivo}"

    if normalizar_aspas:

        # A análise de qualidade realizada no Notebook 03 identificou que
        # os arquivos de 2019 a 2021 possuem milhares de registros em que
        # a linha completa está delimitada externamente por aspas.
        # Por isso, esses arquivos são lidos primeiro como texto, permitindo
        # corrigir esse padrão antes da interpretação das colunas do CSV.
        leitor_texto = spark.read

        if encoding is not None:
            leitor_texto = leitor_texto.option("encoding", encoding)

        df_texto = leitor_texto.text(caminho_arquivo)

        # Obtém o ano a partir do nome da tabela para identificar somente
        # as linhas de dados afetadas pelo padrão de aspas externas.
        ano = nome_tabela.split("_")[-1]

        # Remove as aspas externas do registro e restaura as aspas internas
        # duplicadas, reconstruindo a representação esperada do CSV.
        df_normalizado = (
            df_texto
            .withColumn(
                "value",
                F.when(
                    F.col("value").startswith(f'"{ano}'),
                    F.regexp_replace(
                        F.regexp_replace(
                            F.regexp_replace(
                                F.col("value"),
                                '^"', ''
                            ),
                            '"$', ''
                        ),
                        '""', '"'
                    )
                ).otherwise(F.col("value"))
            )
        )

        # Remove o cabeçalho e interpreta as linhas normalizadas utilizando
        # explicitamente as 21 colunas esperadas. Os valores permanecem como
        # string nesta etapa e serão tipados de forma uniforme na Silver.
        df_bronze = (
            df_normalizado
            .filter(~F.col("value").startswith("Ano,"))
            .select(
                F.from_csv(
                    F.col("value"),
                    schema_bronze_especial_ddl,
                    {
                        "sep": ",",
                        "quote": '"'
                    }
                ).alias("dados")
            )
            .select("dados.*")
        )

    else:

        # Os demais arquivos não apresentam o padrão especial de aspas
        # e continuam utilizando a leitura convencional do CSV.
        leitor = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .option("sep", ",")
        )

        # Aplica encoding específico somente quando necessário.
        if encoding is not None:
            leitor = leitor.option("encoding", encoding)

        df_bronze = leitor.load(caminho_arquivo)

    # Persiste cada período como uma tabela Delta, preservando os nomes
    # originais das colunas e permitindo a atualização do schema em reprocessamentos.
    (
        df_bronze.write
        .format("delta")
        .option("delta.columnMapping.mode", "name")
        .option("overwriteSchema", "true")
        .mode("overwrite")
        .saveAsTable(f"workspace.bronze.{nome_tabela}")
    )

    print(f"Tabela criada: workspace.bronze.{nome_tabela}")

Tabela criada: workspace.bronze.producao_mar_2005_2009
Tabela criada: workspace.bronze.producao_mar_2010_2012
Tabela criada: workspace.bronze.producao_mar_2013_2015
Tabela criada: workspace.bronze.producao_mar_2016_2018
Tabela criada: workspace.bronze.producao_mar_2019
Tabela criada: workspace.bronze.producao_mar_2020
Tabela criada: workspace.bronze.producao_mar_2021
Tabela criada: workspace.bronze.producao_mar_2022
Tabela criada: workspace.bronze.producao_mar_2023
Tabela criada: workspace.bronze.producao_mar_2024
Tabela criada: workspace.bronze.producao_mar_2025


## 2.5 Validação das tabelas Bronze

Após a execução da ingestão, é necessário validar se as tabelas foram corretamente persistidas na camada Bronze.

A validação busca verificar:

- a existência das 11 tabelas esperadas;
- a preservação da quantidade de registros dos arquivos de origem;
- a preservação da estrutura de 21 atributos identificada durante a exploração dos dados.

A investigação detalhada das inconsistências identificadas nos arquivos de 2019, 2020 e 2021 está documentada no notebook `03_qualidade_dados`, tópico **3.2.4 — Consistência dos atributos categóricos**, evitando a repetição dessa análise nesta etapa.

### 2.5.1 Verificação das tabelas criadas

Inicialmente, são listadas as tabelas existentes no schema `workspace.bronze` para confirmar se os 11 arquivos históricos foram persistidos com sucesso.

In [0]:
# Lista as tabelas existentes no schema Bronze
tabelas_bronze = spark.sql("SHOW TABLES IN workspace.bronze")

display(tabelas_bronze)

database,tableName,isTemporary
bronze,producao_mar_2005_2009,false
bronze,producao_mar_2010_2012,false
bronze,producao_mar_2013_2015,false
bronze,producao_mar_2016_2018,false
bronze,producao_mar_2019,false
bronze,producao_mar_2020,false
bronze,producao_mar_2021,false
bronze,producao_mar_2022,false
bronze,producao_mar_2023,false
bronze,producao_mar_2024,false


### 2.5.2 Validação da quantidade de registros

Após confirmar a criação das tabelas, será validada a quantidade de registros armazenados na camada Bronze.

O objetivo desta verificação é confirmar que a ingestão preservou a quantidade de registros existente nos arquivos de origem, sem perda ou inclusão de linhas durante o processamento.

Inicialmente, essa validação era realizada por meio de uma nova leitura dos arquivos RAW utilizando o leitor CSV e da comparação de sua contagem com a respectiva tabela Bronze. Entretanto, a análise de qualidade realizada no notebook `03_qualidade_dados`, tópico **3.2.4 — Consistência dos atributos categóricos**, demonstrou que uma leitura CSV pode manter a quantidade total de registros mesmo quando determinados campos são interpretados incorretamente.

Por esse motivo, a validação foi ajustada para realizar a contagem dos arquivos de origem diretamente como texto, considerando cada linha física do arquivo como um registro e desconsiderando apenas o cabeçalho. Essa abordagem torna a contagem independente da lógica de parsing utilizada durante a ingestão.

Para cada arquivo histórico, a quantidade de registros identificada no arquivo RAW será comparada com a quantidade de registros persistida na respectiva tabela Delta da camada Bronze.

A igualdade entre essas quantidades confirma a preservação do número de registros, mas não é utilizada isoladamente como evidência da correta interpretação dos campos. A estrutura dos dados será validada separadamente nas etapas seguintes.

In [0]:
from pyspark.sql import Row

resultado_validacao = []

for nome_arquivo, nome_tabela, encoding, normalizar_aspas in arquivos_bronze:

    caminho_arquivo = f"{caminho_raw}{nome_arquivo}"

    # O arquivo RAW é lido como texto para que a contagem seja independente
    # das regras de parsing utilizadas durante a ingestão.
    # Cada linha física corresponde a um registro, além da linha de cabeçalho.
    quantidade_raw = (
        spark.read
        .text(caminho_arquivo)
        .count() - 1
    )

    # Conta os registros efetivamente persistidos na respectiva tabela Bronze.
    quantidade_bronze = (
        spark.table(f"workspace.bronze.{nome_tabela}")
        .count()
    )

    resultado_validacao.append(
        Row(
            arquivo=nome_arquivo,
            tabela=nome_tabela,
            registros_raw=quantidade_raw,
            registros_bronze=quantidade_bronze,
            validacao=quantidade_raw == quantidade_bronze
        )
    )

df_validacao_registros = spark.createDataFrame(resultado_validacao)

display(df_validacao_registros)

arquivo,tabela,registros_raw,registros_bronze,validacao
producao-mar-2005-2009.csv,producao_mar_2005_2009,60004,60004,true
producao-mar-2010-2012.csv,producao_mar_2010_2012,37103,37103,true
producao-mar-2013-2015.csv,producao_mar_2013_2015,38758,38758,true
producao-mar-2016-2018.csv,producao_mar_2016_2018,56337,56337,true
producao-mar-2019.csv,producao_mar_2019,22173,22173,true
producao-mar-2020.csv,producao_mar_2020,21815,21815,true
producao-mar-2021.csv,producao_mar_2021,21149,21149,true
producao-mar-2022.csv,producao_mar_2022,19225,19225,true
producao-mar-2023.csv,producao_mar_2023,14824,14824,true
producao_por_poco_2024.csv,producao_mar_2024,15229,15229,true


#### 2.5.2.1 Resultado da validação

A comparação entre os arquivos armazenados no Volume `anp_raw` e as respectivas tabelas da camada Bronze demonstrou que a quantidade de registros foi preservada em todos os períodos analisados.

Não foram identificadas diferenças entre o número de linhas de dados presentes nos arquivos de origem e o número de registros persistidos nas tabelas correspondentes da camada Bronze.

A contagem dos arquivos RAW foi realizada diretamente sobre suas linhas físicas, sem utilizar o parsing CSV. Dessa forma, essa validação é independente das regras utilizadas durante a ingestão e confirma que o processamento preservou a quantidade de registros existente nos arquivos de origem.

Esse resultado constitui uma evidência de integridade da carga em relação à quantidade de registros. Entretanto, a igualdade das contagens não comprova, isoladamente, a correta interpretação dos campos nem a ausência de registros duplicados, aspectos avaliados separadamente nas etapas de validação estrutural e de qualidade dos dados.

### 2.5.3 Validação da estrutura das tabelas Bronze

Além da quantidade de registros, será verificado se a estrutura esperada dos arquivos de origem foi preservada nas tabelas da camada Bronze.

Durante a exploração dos dados, foi identificado que os 11 arquivos históricos possuem a mesma estrutura lógica, composta por **21 atributos**. Dessa forma, será verificada a quantidade de colunas de cada tabela Bronze.

A análise das diferenças de nomenclatura entre os arquivos foi realizada durante a exploração dos dados no Notebook 01. Nesta etapa, a validação concentra-se em verificar se as tabelas Bronze preservaram os 21 atributos esperados. A avaliação detalhada dos tipos de dados e de sua qualidade será realizada posteriormente no notebook `03_qualidade_dados`.

In [0]:
from pyspark.sql import Row

resultado_estrutura = []

# Verifica se todas as tabelas Bronze preservaram
# os 21 atributos identificados nos arquivos de origem.
for nome_arquivo, nome_tabela, encoding, normalizar_aspas in arquivos_bronze:

    df = spark.table(f"workspace.bronze.{nome_tabela}")

    resultado_estrutura.append(
        Row(
            tabela=nome_tabela,
            quantidade_colunas=len(df.columns),
            estrutura_esperada=len(df.columns) == 21
        )
    )

df_validacao_estrutura = spark.createDataFrame(resultado_estrutura)

display(df_validacao_estrutura)

tabela,quantidade_colunas,estrutura_esperada
producao_mar_2005_2009,21,true
producao_mar_2010_2012,21,true
producao_mar_2013_2015,21,true
producao_mar_2016_2018,21,true
producao_mar_2019,21,true
producao_mar_2020,21,true
producao_mar_2021,21,true
producao_mar_2022,21,true
producao_mar_2023,21,true
producao_mar_2024,21,true


#### 2.5.3.1 Resultado da validação estrutural

A validação confirmou que as **11 tabelas da camada Bronze** possuem os **21 atributos** esperados, em conformidade com a quantidade identificada nos arquivos de origem.

Essa verificação confirma a preservação da quantidade de atributos durante a ingestão, mas não é utilizada isoladamente para afirmar a equivalência dos nomes ou dos tipos de dados entre os diferentes períodos.

Dessa forma, as validações realizadas nesta etapa confirmam a persistência das tabelas e a preservação da quantidade de registros e atributos esperados, fornecendo a base para as etapas posteriores de avaliação da qualidade, tratamento e padronização dos dados.

## 2.6 Conclusão da camada Bronze

Nesta etapa foi construída a primeira camada do Lakehouse seguindo os princípios da Arquitetura Medalhão.

Os arquivos históricos de produção offshore da ANP foram ingeridos a partir do Volume `anp_raw` e persistidos como **11 tabelas Delta** no schema `workspace.bronze`, preservando a organização dos dados por período e mantendo sua estrutura o mais próxima possível da fonte original.

Durante a ingestão foram aplicadas configurações técnicas necessárias para a correta interpretação e persistência dos arquivos. Entre elas estão a definição do *encoding* para determinados períodos, a utilização do recurso **Column Mapping** do Delta Lake para preservar os nomes originais das colunas e a normalização do padrão específico de aspas identificado nos arquivos de **2019, 2020 e 2021**.

A necessidade dessa última regra foi identificada durante a avaliação de qualidade realizada no notebook `03_qualidade_dados`, tópico **3.2.4 — Consistência dos atributos categóricos**. A investigação demonstrou que o padrão de formatação desses arquivos provocava a interpretação incorreta de determinados registros durante o parsing convencional do CSV. A correção foi incorporada à etapa de ingestão, mantendo os arquivos originais da camada RAW inalterados.

As validações da camada Bronze confirmaram a criação das tabelas esperadas, a preservação da quantidade de registros dos arquivos de origem e a presença dos **21 atributos** esperados nas tabelas persistidas.

Com a camada Bronze estruturada e validada, os dados estão preparados para a etapa de **avaliação da qualidade dos dados** e, posteriormente, para as transformações de padronização e consolidação que resultarão na camada Silver.